# bonus a - batching

in lab 6 the network updated all 62,006 weights **after every single image**: 35,000 optimizer steps an epoch, each one based on one noisy example. **batching** shows the network N images, averages their gradients, and takes **one** step.

- the gradient is less noisy (its an average of N), so a bigger learning rate is ok
- N times fewer optimizer steps
- the work comes in bigger chunks: one tall matrix multiply instead of N short ones. thats what gpus need to keep thousands of cores busy (cudagrad trains with batches of 128)

with layouts, adding a batch is almost free: **the batch is just one more mode.**

same deal as the main lab: `# todo:` cells are yours, `# given code` cells are already written, and if youre stuck put `%load solutions/NAME.py` in the cell and run it twice. the formula sheet and python stuff for this notebook are right below.

## formula sheet

- a batch of $N$ images is one tensor $(N, C, H, W)$ with row-major strides $(CHW,\; HW,\; W,\; 1)$. pixel $(n, c, y, x)$ sits at $nCHW + cHW + yW + x$
- batched im2col: rows $(n, oy, ox)$ with strides $(CHW, W, 1)$, columns $(c, i, j)$ with strides $(HW, W, 1)$
- conv for the whole batch is one matrix multiply $Y = F X^T$ where $X$ has $N \cdot OH \cdot OW$ rows (every patch of every image) and $C K^2$ columns
- the batch loss is the average: $L = \dfrac{1}{N} \sum_n L_n$, so the gradient is the average too: $\dfrac{\partial L}{\partial w} = \dfrac{1}{N} \sum_n \dfrac{\partial L_n}{\partial w}$
- softmax + cross-entropy over a batch: $dz[n][k] = \big(p[n][k] - [k = y_n]\big) / N$
- fully connected over a batch: $Y = X W^T + b$ (the same $b$ added to every row), $\;dX = dY\,W$, $\;dW = dY^T X$, $\;db = \sum_n dY[n]$
- broadcasting: a stride of $0$ repeats memory. $(N, n_\text{out}) : (0, 1)$ reads the same bias row $N$ times, $(N, C_\text{out}, P) : (0, 1, 0)$ repeats bias $k$ over every image and pixel
- a weight gradient is a sum over the batch: for conv1 with batch $N$ thats $K = N \cdot 28 \cdot 28$ products per weight ($100{,}352$ when $N = 128$)
- steps per epoch $= \lceil \text{images} / N \rceil$. with fewer steps you can usually raise the learning rate, roughly in proportion to $N$ (the "linear scaling rule", try a few)

## python stuff for this notebook
- `isinstance(layer, Conv)` checks what kind of thing `layer` is. `forward_batched` uses it to pick the batched version of every layer
- `-(-a // b)` is `a / b` rounded **up** (floor division of a negative rounds the other way): `-(-7 // 2)` is 4
- `lambda: x.add_grad(...)` is a one line function with no arguments, perfect for a short `backward_fn`
- a function with `yield` (like `batches`) hands out one batch at a time
- `time.perf_counter()` twice and subtract = how long something took
- `f"{v:5.1f}"` is a number 5 characters wide with 1 decimal

In [ ]:
import time
from babytorch import *          # everything from the main lab (the answer key versions)
import viz, checks

### a.1 im2col with a batch mode
a batch of N images stored one after another is the layout `(N, C, H, W):(C·H·W, H·W, W, 1)`. for the convolution, every output pixel of *every image* turns into a row of the patch matrix: rows `(n, oy, ox)`, columns `(c, i, j)` exactly like before. for lenet's first layer with N = 64 thats a 50,176 x 75 patch matrix, and it still never gets built.

In [ ]:
# todo: batched_im2col_layout   (stuck? put  %load solutions/batched_im2col_layout.py  in this cell and run it twice)
def batched_im2col_layout(N, C, H, W, K):
    """im2col for a whole batch stored (N, C, H, W) row-major, so strides (C*H*W, H*W, W, 1).

    rows    = (n, oy, ox): every output pixel of every image  -> mode (N, OH, OW)
    columns = (c, i, j):   a spot in the patch, same as before"""
    OH, OW = H - K + 1, W - K + 1
    # how to think about it:
    #   its im2col_layout from lab 2 with one more letter. pixel (n, c, oy + i, ox + j) of the batch is at
    #       n*C*H*W + c*H*W + (oy + i)*W + (ox + j)
    #   multiply it out:   n*C*H*W + oy*W + ox*1   +   c*H*W + i*W + j*1
    #   the rows are (n, oy, ox) and the columns are (c, i, j). the number in front of each letter is its
    #   stride. the columns didnt change at all from lab 2
    # stuff youll need:
    #   Layout(shape, stride) with nested tuples. the shape is already filled in, replace every ...
    return Layout(((N, OH, OW), (C, K, K)),
                  ((..., ..., ...), (..., ..., ...)))

In [ ]:
checks.batched_im2col_layout(batched_im2col_layout)

In [ ]:
viz.show_layout(batched_im2col_layout(2, 1, 3, 3, 2),
                title="2 images of 1x3x3 with 2x2 patches: rows = (image, oy, ox), columns = (c, i, j)")

### a.2 broadcasting the bias: stride 0
a fully connected layer on a batch works out $Y = XW^T$ (N x out) and then adds the *same* bias row onto every one of the N rows. instead of copying the bias N times, read it thru a layout whose row stride is **0**.

In [ ]:
# todo: broadcast   (stuck? put  %load solutions/broadcast.py  in this cell and run it twice)
def broadcast(n_rows, n_cols):
    """reads a list of n_cols numbers as an n_rows x n_cols matrix where every row is that same list.
    a stride of 0 means stepping that way doesnt move in memory"""
    # how to think about it:
    #   stepping right (to the next column) should move to the next number in memory, so the column stride is 1.
    #   stepping down (to the next row) should land on the SAME numbers again. so how far should one step
    #   down move in memory?
    # stuff youll need:
    #   Layout((n_rows, n_cols), (row_stride, column_stride)), replace the ...
    return Layout((n_rows, n_cols), (..., 1))

In [ ]:
checks.broadcast(broadcast)

In [ ]:
print_layout(broadcast(4, 3))      # 4 rows, all reading offsets 0, 1, 2

### a.3 the batched layers
heres every layer for a batch. look at `conv2d_batched` closely:
- **forward**: one gather + one matmul for the whole batch, then a gather to reorder `(Cout, N, P)` → `(N, Cout, P)`, and the bias repeated with a stride-0 layout `(N, Cout, P):(0, 1, 0)`
- **backward**: the weight gradient is now `matmul_t(dy_rows, ...)` with rows of length **N·P**, a sum over every image and every position. remember that one for bonus b

max-pool needs nothing new (a batch of N images with C channels is just N·C channels) and relu is the same function as before.

In [ ]:
# given code - read it, then run it
#
# what this does: every layer, but for a whole batch at a time
# stuff in here:
#   N, C, H, W = x.shape              the batch size is the new first number
#   conv2d_batched                    one gather + ONE matmul for every image in the batch. the matmul gives the
#                                     maps back as (Cout, N, P), so a gather with Layout((N, Cout, P), (P, N*P, 1))
#                                     reads them in (N, Cout, P) order without moving anything by hand
#   Layout((N, Cout, P), (0, 1, 0))   stride 0 for n and p: bias[k] gets repeated over every image and pixel
#   y.backward_fn = lambda: ...       a backward_fn can be a lambda when its just one line
#   maxpool2d_batched                 a batch of N images with C channels is just N*C channels, nothing new
#   linear_batched                    Y = X W^T + b.  dW = dY^T X,  dX = dY W,  db = the sum over the batch
#   softmax_cross_entropy_batched     the AVERAGE loss, so every image's gradient gets divided by N
#   forward_batched                   walks the model's layers and calls the batched version of each one
#   isinstance(layer, Conv)           checks what kind of layer it is
#   batches(loader, n)                a generator (it uses yield) that groups (image, label) pairs into lists of n
#
def conv2d_batched(x, weight, bias):
    """x: (N, C, H, W) -> (N, Cout, OH, OW), with ONE matrix multiply for the whole batch"""
    N, C, H, W = x.shape
    Cout, _, K, _ = weight.shape
    OH, OW = H - K + 1, W - K + 1
    P, n = OH * OW, C * K * K
    L = batched_im2col_layout(N, C, H, W, K)
    patches = rows(gather(x.data, L), n)                        # N*P rows
    out = matmul_t(rows(weight.data, n), patches)               # Cout x (N*P), stored as "C, N, H, W"
    nchw = gather(out, Layout((N, Cout, P), (P, N * P, 1)))     # read it back in N, C, H, W order
    b = gather(bias.data, Layout((N, Cout, P), (0, 1, 0)))      # bias[k] repeated over n and p (stride 0)
    y = Tensor([v + bb for v, bb in zip(nchw, b)], (N, Cout, OH, OW), (x, weight, bias), "conv2d_batched")

    def backward_fn():
        dy = gather(y.grad, Layout((Cout, N, P), (P, Cout * P, 1)))   # back to Cout x (N*P)
        dy_rows = rows(dy, N * P)
        bias.add_grad([sum(r) for r in dy_rows])
        # the weight gradient: Cout x n answers, each one a sum over K = N*P things -> thats bonus b (split-k)
        weight.add_grad(matmul_t(dy_rows, rows(gather(x.data, transpose(L)), N * P)))
        if x.requires_grad:
            dy_cols = rows(gather(dy, Layout((N * P, Cout), (1, N * P))), Cout)
            w_cols = rows(gather(weight.data, Layout((n, Cout), (1, n))), Cout)
            dx = [0.0] * len(x.data)
            for o, g in zip(offsets(L), matmul_t(dy_cols, w_cols)):
                dx[o] += g
            x.add_grad(dx)
    y.backward_fn = backward_fn
    return y


def maxpool2d_batched(x, p=2):
    """batching max-pool is free: an (N, C, H, W) batch is just N*C channels of H x W"""
    N, C, H, W = x.shape
    y = Tensor(maxpool2d_forward(x.data, N * C, H, W, p), (N, C, H // p, W // p), (x,), "maxpool2d_batched")
    y.backward_fn = lambda: x.add_grad(maxpool2d_backward(y.grad, x.data, N * C, H, W, p))
    return y


def flatten_batched(x):
    """(N, C, H, W) -> (N, C*H*W): the same memory, a new shape"""
    N = x.shape[0]
    y = Tensor(x.data, (N, len(x.data) // N), (x,), "flatten_batched")
    y.backward_fn = lambda: x.add_grad(y.grad)
    return y


def linear_batched(x, weight, bias):
    """x: (N, n_in) -> (N, n_out).   Y = X W^T + b, with the bias repeated down the batch"""
    N, n_in = x.shape
    n_out = bias.shape[0]
    X = rows(x.data, n_in)
    out = matmul_t(X, rows(weight.data, n_in))                  # N x n_out
    y = Tensor([v + b for v, b in zip(out, gather(bias.data, broadcast(N, n_out)))], (N, n_out),
               (x, weight, bias), "linear_batched")

    def backward_fn():
        dY_cols = rows(gather(y.grad, Layout((n_out, N), (1, n_out))), N)     # n_out rows of N
        bias.add_grad([sum(c) for c in dY_cols])                              # summed over the batch
        X_cols = rows(gather(x.data, Layout((n_in, N), (1, n_in))), N)        # n_in rows of N
        weight.add_grad(matmul_t(dY_cols, X_cols))                            # summed over the batch
        if x.requires_grad:
            W_cols = rows(gather(weight.data, Layout((n_in, n_out), (1, n_in))), n_out)
            x.add_grad(matmul_t(rows(y.grad, n_out), W_cols))                 # N x n_in
    y.backward_fn = backward_fn
    return y


def softmax_cross_entropy_batched(logits, labels):
    """the AVERAGE cross-entropy over the batch (so the learning rate doesnt depend on the batch size)"""
    N, n_cls = logits.shape
    results = [cross_entropy_loss(z, lab) for z, lab in zip(rows(logits.data, n_cls), labels)]
    out = Tensor([sum(loss for loss, _ in results) / N], (1,), (logits,), "softmax_cross_entropy_batched")

    def backward_fn():
        g = out.grad[0] / N
        logits.add_grad([v * g for (_, probs), lab in zip(results, labels)
                         for v in cross_entropy_backward(probs, lab)])
    out.backward_fn = backward_fn
    return out


def forward_batched(model, x):
    """runs a babytorch CNN (made of Conv / MaxPool / Flatten / Linear) on a whole batch (N, 3, 32, 32)"""
    for layer in model.layers:
        if isinstance(layer, Conv):
            x = conv2d_batched(x, layer.weight, layer.bias)
            x = relu(x) if layer.use_relu else x
        elif isinstance(layer, MaxPool):
            x = maxpool2d_batched(x, layer.p)
        elif isinstance(layer, Flatten):
            x = flatten_batched(x)
        elif isinstance(layer, Linear):
            x = linear_batched(x, layer.weight, layer.bias)
            x = relu(x) if layer.use_relu else x
        else:
            raise TypeError(f"dont know how to batch a {type(layer).__name__}")
    return x


def stack(images):
    """a list of N images (3,072 floats each) -> one (N, 3, 32, 32) tensor"""
    return Tensor([v for image in images for v in image], (len(images), 3, 32, 32))


def batches(loader, batch_size):
    """groups a DataLoader's (image, label) pairs into lists of batch_size"""
    images, labels = [], []
    for image, label in loader:
        images.append(image)
        labels.append(label)
        if len(images) == batch_size:
            yield images, labels
            images, labels = [], []
    if images:
        yield images, labels

### a.4 batching doesnt change the math
check it: the batched forward pass gives exactly the one-at-a-time logits, and the batched gradient is exactly the **average** of the one-at-a-time gradients.

In [ ]:
test_loader = DataLoader(DATA / "test.bin")
images = [test_loader.image(k) for k in range(0, 150, 10)]       # 15 test images, one of every 10
labels = [test_loader.labels[k] for k in range(0, 150, 10)]
model = lenet5(seed=4)

batched = forward_batched(model, stack(images)).data
one_by_one = [v for image in images for v in model(Tensor(image, (3, 32, 32))).data]
print("biggest |batched - one at a time| =", max(abs(a - b) for a, b in zip(batched, one_by_one)))

loss = softmax_cross_entropy_batched(forward_batched(model, stack(images)), labels)
loss.backward()
g_batched = [list(p.grad) for p in model.parameters()]
for p in model.parameters():
    p.grad = None
for image, label in zip(images, labels):
    softmax_cross_entropy(model(Tensor(image, (3, 32, 32))), label).backward()    # gradients pile up
g_avg = [[g / len(images) for g in p.grad] for p in model.parameters()]
print("biggest |batched gradient - average of single gradients| =",
      max(abs(a - b) for x, y in zip(g_batched, g_avg) for a, b in zip(x, y)))

### a.5 a batched training step

In [ ]:
# todo: train_step_batched   (stuck? put  %load solutions/train_step_batched.py  in this cell and run it twice)
def train_step_batched(model, opt, images, labels):
    """one sgd step on a whole batch. gives back (average loss, how many guesses were right)"""
    # how to think about it:
    #   its train_step from lab 6, but for a whole batch at once:
    #   1. glue the images into one (N, 3, 32, 32) tensor with stack(images) and run forward_batched(model, it).
    #      you get logits with shape (N, 10), one row of 10 scores per image
    #   2. loss = softmax_cross_entropy_batched(logits, labels). its already the average over the batch
    #   3. zero_grad, backward, step. ONE optimizer step for the whole batch, thats the whole point
    #   4. count the right guesses: cut logits.data into rows of 10, take the argmax of every row, and count
    #      how many match their label
    #   give back  (loss.data[0], how many were right)
    # stuff youll need:
    #   stack(images), forward_batched(model, x), softmax_cross_entropy_batched(logits, labels)   (given above)
    #   logits.shape[1]                     how many classes (10)
    #   rows(logits.data, n_cls)            one row of scores per image
    #   max(range(n_cls), key=r.__getitem__)   the argmax of one row r
    #   sum(g == lab for g, lab in zip(guesses, labels))   True counts as 1, so sum() counts the matches
    raise NotImplementedError("todo: train_step_batched")

In [ ]:
checks.train_step_batched(train_step_batched)

In [ ]:
# given code - read it, then run it
#
# what this does: the training loop from lab 6, but one sgd step per batch instead of one per image
# stuff in here:
#   -(-len(loader) // batch_size)     batches per epoch, rounded up (the last batch can be smaller)
#   run_loss += loss * len(images)    loss is an average, so multiply it back to get the batch's total
#   opt.t % log_every == 0            prints a line every log_every steps
#
def train_batched(model, loader, epochs=1, batch_size=16, lr_max=0.02, lr_min=1e-5, log_every=50):
    """like babytorch.train, but one sgd step per batch. gives back images per second"""
    steps = epochs * -(-len(loader) // batch_size)
    opt = SGD(model.parameters(), lr_max, lr_min, total_steps=steps, schedule=cosine_annealing_lr)
    start, seen = time.time(), 0
    for epoch in range(1, epochs + 1):
        run_loss, run_right, n = 0.0, 0, 0
        for images, labels in batches(loader, batch_size):
            loss, right = train_step_batched(model, opt, images, labels)
            run_loss += loss * len(images)
            run_right += right
            n += len(images)
            seen += len(images)
            if opt.t % log_every == 0 or n == len(loader):
                print(f"epoch {epoch}/{epochs}  image {n:>6}/{len(loader)}  loss {run_loss / n:.3f}  "
                      f"train acc {run_right / n:6.1%}  lr {opt.lr:.5f}  ({seen / (time.time() - start):.0f} images/s)",
                      flush=True)
    return seen / (time.time() - start)

### a.6 is it faster?
time one training step per image for a few batch sizes, against the main lab's `train_step`.

In [ ]:
loader = DataLoader(DATA / "train.bin", shuffle=True, per_class=20)
data = [pair for _, pair in zip(range(64), loader)]                  # 64 training pairs
ms = {}
for B in (1, 4, 16, 64):
    m = lenet5(seed=1)
    opt = SGD(m.parameters(), 0.01)
    imgs, labs = [d[0] for d in data[:B]], [d[1] for d in data[:B]]
    train_step_batched(m, opt, imgs, labs)                                # warm up (builds the offset tables)
    t = time.perf_counter()
    for _ in range(64 // B):                                              # 64 images worth of steps
        train_step_batched(m, opt, imgs, labs)
    ms[B] = (time.perf_counter() - t) / 64 * 1000
m = lenet5(seed=1)
opt = SGD(m.parameters(), 0.01)
t = time.perf_counter()
for image, label in data:
    train_step(m, opt, image, label)
core_ms = (time.perf_counter() - t) / 64 * 1000
print(f"main lab, one image at a time: {core_ms:.1f} ms per image")
for B, v in ms.items():
    print(f"batch of {B:>2}: {v:5.1f} ms per image   ({core_ms / v:.2f}x)")

talk about it: why so little? in plain python every multiply-add still costs the same, batched or not. batching only saves the stuff *around* the math: the optimizer step (62,006 weights updated once per batch instead of once per image) and some per-call bookkeeping. on a **gpu** its a whole different story. one image gives the first conv only 784 x 6 outputs to compute, nowhere near enough to keep thousands of cores busy. a batch of 128 gives 128x more independent work for almost free. thats why every gpu library trains in batches.

### a.7 does it learn as well?
same data and the same number of epochs, one image at a time vs batches of 16. bigger batches mean fewer, smoother steps, so they need a bigger learning rate (8x bigger here).

In [ ]:
train_loader = DataLoader(DATA / "train.bin", shuffle=True, per_class=200)
val_loader = DataLoader(DATA / "val.bin")
results = {}
for B, lr in [(1, 0.005), (16, 0.04)]:
    random.seed(0)
    m = lenet5(seed=0)
    t = time.time()
    if B == 1:
        train(m, train_loader, epochs=2, lr_max=lr, log_every=1000)
    else:
        train_batched(m, train_loader, epochs=2, batch_size=B, lr_max=lr, log_every=50)
    secs = time.time() - t
    val_acc, _ = evaluate(m, val_loader, limit=500)
    test_acc, _ = evaluate(m, test_loader)
    results[B] = (secs, val_acc, test_acc)
for B, (secs, val_acc, test_acc) in results.items():
    print(f"batch {B:>2}: {secs:5.0f} s   val {val_acc:.1%}   real test {test_acc:.1%}")

### a.8 matrix shapes with a batch
every layer is a matrix multiply $C_{M\times N} = A_{M\times K}\,B^T$. look at the **weight gradient** ones: the output is tiny (just the filters) but **K**, how many things get added up for each output, grows with the batch.

In [ ]:
N = 128   # cudagrad's batch size
rows_ = [("conv1 forward", 6, N * 28 * 28, 75), ("conv1 weight gradient", 6, 75, N * 28 * 28),
         ("conv2 forward", 16, N * 10 * 10, 150), ("conv2 weight gradient", 16, 150, N * 10 * 10),
         ("fc1 forward", N, 120, 400), ("fc1 weight gradient", 120, 400, N)]
print(f"{'matmul':<24}{'M':>7}{'N':>8}{'K':>9}   outputs   added up per output")
for name, M_, N_, K_ in rows_:
    print(f"{name:<24}{M_:>7}{N_:>8}{K_:>9}   {M_ * N_:>7,}   {K_:>10,}")

`conv1 weight gradient`: only **450 outputs**, but each one is a sum of **100,352 products**. on a gpu thats 450 threads doing super long loops while thousands of cores sit around doing nothing. the fix is **split-k**, thats [bonus b](addon_B_split_k.ipynb).